# [Chapter 9 — Practical Issues in Fitting, §9.6] Pitfall 6: Distinguishing Model Uncertainty from Real-World Uncertainty

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 9 — Practical Issues in Fitting
**Considerations developed:** 9, 13
**Estimated runtime:** ≤ 20s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Conceptual notebook: model uncertainty (parameters within a fixed model class) and real-world uncertainty (model misspecification) are distinct sources of error. Confidence intervals from fitting capture only the first.

## What you should already know
Chapter 8 fitting notebooks.


## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from shared import book_style, BOOK_COLORS, integrate_sir_i
from shared.parameters import baseline_chapter_08
from shared.seeds import set_seed_chapter_08
from shared.verification import assert_within_tolerance
set_seed_chapter_08()
book_style()


## Two sources of uncertainty

When a fit produces a 95% CI for $\mathcal{R}_0$ as $[1.8, 2.2]$, this represents **parameter uncertainty within the assumed model class**. It does not represent:
- Whether the SIR model is the right model class (model misspecification)
- Whether the assumed observation model (Poisson? Gaussian?) is correct
- Whether parameters are constant in time (often violated)

The book's convention: report CIs but flag the model assumptions explicitly.

In [ ]:
# Demonstration: same data, two models, different "best" R_0
params = baseline_chapter_08()
result = integrate_sir_i(params)
t, S, I = result['t'], result['S'], result['I']
J = params['c_I'] * params['beta'] * (S/params['N']) * I
J_obs = np.maximum(J + 0.05 * J.max() * np.random.randn(len(t)), 0)

# Model A: SIR (correct)
# Model B: SEIR (wrongly assumed)
# We won't actually fit SEIR here, but illustrate that the same data fits multiple models with different R_0

mask = (t > 5) & (t < 20)

# Within-model parameter uncertainty (illustrate via bootstrap)
n_boot = 200
R0_boot = []
for _ in range(n_boot):
    idx = np.random.randint(0, mask.sum(), mask.sum())
    a_h = J_obs[mask][idx].mean() / I[mask][idx].mean()
    R0_boot.append(a_h * params['tau_R'] / (S[mask].mean() / params['N']))
R0_boot = np.array(R0_boot)

ci_low, ci_high = np.percentile(R0_boot, [2.5, 97.5])
print(f"Bootstrap 95% CI on R_0 (within-SIR-model): [{ci_low:.2f}, {ci_high:.2f}]")
print()
print("BUT — this CI assumes the SIR model is correct.")
print("If the true generating process were SEIR (with a latent period), the same incidence data")
print("would imply a DIFFERENT R_0 because of the different generation interval.")
print()
print("Chapter 9 §9.6 catalogues this distinction. Real-world uncertainty is model-class uncertainty;")
print("parameter CIs from a single model class understate the true uncertainty.")
